## RST label counts and significance tests (leaf-level EDUs)
 Part 1: absolute numbers, proportions and percentages per label  
 Part 2: significance tests, Conflict vs. No Conflict

In [1]:
# Step 0: imports
import pandas as pd
from pathlib import Path
from scipy.stats import fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests   # pip install statsmodels

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

In [2]:
# Step 1: load the corpus (same as for barplots in 03_analysis_labelsdistribution.ipynb)
csv_path = Path("../data/output/main_conflicts_aligned_paragraphid.csv")
df = pd.read_csv(csv_path, index_col=0)

# "_" means: this EDU carries no Conflict annotation
df["Conflict_Type"] = df["Conflict_Type"].replace("_", "No_Conflict")

In [3]:
# Step 2: prepare the columns we need

# 2a) merge -N / -S variants (as in drop_relations() function in 03_analysis_labelsdistribution.ipynb)
df["relation"] = df["rstree_relation_leaf"].replace({
    "reason-N": "reason", 
    "reason-S": "reason",
    "evaluation-N": "evaluation", 
    "evaluation-S": "evaluation"})

# 2b) binary Conflict column (all four types -> "Conflict")
conflict_types = ["Challenge", "Correction", "Direct_NegEval", "Indirect_NegEval"]
df["Conflict_Binary"] = df["Conflict_Type"].isin(conflict_types).map({True: "Conflict", False: "No_Conflict"})

# 2c) relations to report (=index_list in 03_analysis_labelsdistribution.ipynb; span, sameunit, unless etc. are left out)
relations = [
    "attribution",
    "background", "enablement", "evaluation", "evidence", "motivation", "reason",  # pragmatic
    "antithesis", "concession", "contrast",                                         # contrastive
    "cause", "circumstance", "condition", "e-elaboration", "elaboration",
    "interpretation", "means", "purpose", "result",                                 # semantic
    "preparation", "restatement", "summary",                                        # textual
    "list", "conjunction", "joint", "sequence",                                     # multinuclear
]


## Part 1: counts, proportions, percentages

In [4]:
# Step 3: function that builds the table for any grouping column
def label_table(df, group_col, relations):
    """
    For every group (e.g. each Conflict type) and every relation:
      n          = absolute number of EDUs with this relation
      proportion = n / all EDUs of the group      (normalised number, 0-1)
      percent    = proportion * 100               (normalised percentage)
    The denominator is ALL EDUs of the group (also those with span, sameunit, ...),
    so the percentages are identical to the ones in your barplots.
    """
    counts_all = pd.crosstab(df["relation"], df[group_col])   # rows: relations, columns: groups
    totals = counts_all.sum()                                 # number of EDUs per group

    counts = counts_all.reindex(relations).fillna(0).astype(int)
    proportion = counts / totals
    percent = proportion * 100

    table = pd.concat(
        {"n": counts, "proportion": proportion.round(4), "percent": percent.round(2)},
        axis=1,
    )
    return table, totals

In [22]:
# Step 4: table for the five labels (Challenge, Correction, ..., No_Conflict)
table_types, totals_types = label_table(df, "Conflict_Type", relations)
print("""EDUs per Conflict type:
    n is the absolute count; 
    proportion is the count divided by all EDUs of that group (0 to 1);
    percent is the same ×100
""")
totals_types

table_types.to_csv("label_counts_conflict_types.csv")
table_types

EDUs per Conflict type:
    n is the absolute count; 
    proportion is the count divided by all EDUs of that group (0 to 1);
    percent is the same ×100



n                                                        proportion                                                          percent                                                       
Conflict_Type  Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict  Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict
relation                                                                                                                                                                                                          
attribution           15          9             22               17         195     0.1829     0.0629         0.0282           0.0381      0.0491     18.29       6.29           2.82             3.81        4.91
background             1          0              3                0          54     0.0122     0.0000         0.0038           0.0000      0.0136      1.22       0.00           0.38             0.00        1.36
enablement             0          0              2                0           5     0.0000     0.0000         0.0026           0.0000      0.0013      0.00       0.00           0.26             0.00        0.13
evaluation             0          1             16                3          61     0.0000     0.0070         0.0205           0.0067      0.0154      0.00       0.70           2.05             0.67        1.54
evidence               1          1              4                5          39     0.0122     0.0070         0.0051           0.0112      0.0098      1.22       0.70           0.51             1.12        0.98
motivation             0          0              0                0          11     0.0000     0.0000         0.0000           0.0000      0.0028      0.00       0.00           0.00             0.00        0.28
reason                 0          0             11                4          53     0.0000     0.0000         0.0141           0.0090      0.0134      0.00       0.00           1.41             0.90        1.34
antithesis             4          5              6                4          33     0.0488     0.0350         0.0077           0.0090      0.0083      4.88       3.50           0.77             0.90        0.83
concession             1          1             17                6          59     0.0122     0.0070         0.0218           0.0135      0.0149      1.22       0.70           2.18             1.35        1.49
contrast               2          2              8                2           8     0.0244     0.0140         0.0102           0.0045      0.0020      2.44       1.40           1.02             0.45        0.20
cause                  0          2              6                0          14     0.0000     0.0140         0.0077           0.0000      0.0035      0.00       1.40           0.77             0.00        0.35
circumstance           0          0             19                9          83     0.0000     0.0000         0.0243           0.0202      0.0209      0.00       0.00           2.43             2.02        2.09
condition              0          1              7                3          43     0.0000     0.0070         0.0090           0.0067      0.0108      0.00       0.70           0.90             0.67        1.08
e-elaboration         11         18             98               65         427     0.1341     0.1259         0.1255           0.1457      0.1076     13.41      12.59          12.55            14.57       10.76
elaboration            3          5             28               12         191     0.0366     0.0350         0.0359           0.0269      0.0481      3.66       3.50           3.59             2.69        4.81
interpretation         0          0              1                2          14     0.0000     0.0000         0.0013           0.0045      0.0035      0.00       0.00           0.13             0.45        0.35
means         

In [24]:
# Step 5: table for Conflict vs. No Conflict
table_binary, totals_binary = label_table(df, "Conflict_Binary", relations)
print("""EDUs per "group" (Confl vs. No Confl):
    n is the absolute count; 
    proportion is the count divided by all EDUs of that group (0 to 1);
    percent is the same ×100
""")
totals_binary

table_binary.to_csv("label_counts_binary.csv")
table_binary

EDUs per "group" (Confl vs. No Confl):
    n is the absolute count; 
    proportion is the count divided by all EDUs of that group (0 to 1);
    percent is the same ×100



n             proportion              percent            
Conflict_Binary Conflict No_Conflict   Conflict No_Conflict Conflict No_Conflict
relation                                                                        
attribution           63         195     0.0434      0.0491     4.34        4.91
background             4          54     0.0028      0.0136     0.28        1.36
enablement             2           5     0.0014      0.0013     0.14        0.13
evaluation            20          61     0.0138      0.0154     1.38        1.54
evidence              11          39     0.0076      0.0098     0.76        0.98
motivation             0          11     0.0000      0.0028     0.00        0.28
reason                15          53     0.0103      0.0134     1.03        1.34
antithesis            19          33     0.0131      0.0083     1.31        0.83
concession            25          59     0.0172      0.0149     1.72        1.49
contrast              14           8     0.0096      0.0020     0.96        0.20
cause                  8          14     0.0055      0.0035     0.55        0.35
circumstance          28          83     0.0193      0.0209     1.93        2.09
condition             11          43     0.0076      0.0108     0.76        1.08
e-elaboration        192         427     0.1322      0.1076    13.22       10.76
elaboration           48         191     0.0331      0.0481     3.31        4.81
interpretation         3          14     0.0021      0.0035     0.21        0.35
means                  4          10     0.0028      0.0025     0.28        0.25
purpose               46         109     0.0317      0.0275     3.17        2.75
result                17          24     0.0117      0.0060     1.17        0.60
preparation            5          52     0.0034      0.0131     0.34        1.31
restatement            4          40     0.0028      0.0101     0.28        1.01
summary                0           3     0.0000      0.0008     0.00        0.08
list                  46         157     0.0317      0.0396     3.17        3.96
conjunction          202         555     0.1391      0.1398    13.91       13.98
joint                  7          25     0.0048      0.0063     0.48        0.63
sequence               9          30     0.0062      0.0076     0.62        0.76

## Part 2: significance tests
For each relation X we ask: is the share of EDUs with relation X different
between group A and group B? This gives a 2x2 table per relation:

 |            | has X | does not have X |
 |------------|-------|-----------------|
 | group A    |  a    |  A_total - a    |
 | group B    |  b    |  B_total - b    |

 We test it with Fisher's exact test (also reliable for small groups such as Challenge).  
 Because we test ~26 relations, the p-values are corrected for multiple testing (Holm).


In [10]:
# Step 6: function for the two-group comparison
def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."

def compare_two_groups(df, group_col, group_a, group_b, relations):
    a = df[df[group_col] == group_a]
    b = df[df[group_col] == group_b]

    rows = []
    for rel in relations:
        a_with = (a["relation"] == rel).sum()
        b_with = (b["relation"] == rel).sum()
        a_without = len(a) - a_with
        b_without = len(b) - b_with

        odds_ratio, p = fisher_exact([[a_with, a_without], [b_with, b_without]])

        rows.append({
            "relation": rel,
            f"n_{group_a}": a_with,
            f"%_{group_a}": round(100 * a_with / len(a), 2),
            f"n_{group_b}": b_with,
            f"%_{group_b}": round(100 * b_with / len(b), 2),
            "diff_pct_points": round(100 * (a_with / len(a) - b_with / len(b)), 2),
            "odds_ratio": round(odds_ratio, 2),   # > 1: more frequent in group A
            "p": p,
        })

    res = pd.DataFrame(rows)
    # Holm correction across all tested relations (use method="fdr_bh" for a less strict one)
    res["p_corrected"] = multipletests(res["p"], method="holm")[1]
    res["sig"] = res["p_corrected"].apply(stars)
    return res


In [25]:
# %% Step 7: Conflict vs. No Conflict (binary)
res_binary = compare_two_groups(df, "Conflict_Binary", "Conflict", "No_Conflict", relations)
res_binary.to_csv("significance_binary.csv", index=False)

#print(res_binary.round({"p": 5, "p_corrected": 5}).to_string(index=False))
res_binary_rounded = res_binary.round({"p": 5, "p_corrected": 5})
print("""diff_pct_points is the difference between the Conflict vs. No Conflict in percentage points,
odds_ratio above 1 means the relation is more frequent in Conflicts,
p_corrected is the Holm-corrected p-value.""")
res_binary_rounded

diff_pct_points is the difference between the Conflict vs. No Conflict in percentage points,
odds_ratio above 1 means the relation is more frequent in Conflicts


,relation,n_Conflict,%_Conflict,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,63,4.34,195,4.91,-0.57,0.88,0.42809,1.00000,n.s.
1,background,4,0.28,54,1.36,-1.09,0.20,0.00025,0.00652,**
2,enablement,2,0.14,5,0.13,0.01,1.09,1.00000,1.00000,n.s.
3,evaluation,20,1.38,61,1.54,-0.16,0.89,0.80040,1.00000,n.s.
4,evidence,11,0.76,39,0.98,-0.23,0.77,0.52274,1.00000,n.s.
5,motivation,0,0.00,11,0.28,-0.28,0.00,0.04353,0.87058,n.s.
6,reason,15,1.03,53,1.34,-0.30,0.77,0.41171,1.00000,n.s.
7,antithesis,19,1.31,33,0.83,0.48,1.58,0.11661,1.00000,n.s.
8,concession,25,1.72,59,1.49,0.24,1.16,0.53568,1.00000,n.s.
9,contrast,14,0.96,8,0.20,0.76,4.82,0.00031,0.00772,**


In [18]:
# %% Step 8: each Conflict type vs. No Conflict
for ct in conflict_types:
    print(f"\n===== {ct} vs. No_Conflict =====")
    res = compare_two_groups(df, "Conflict_Type", ct, "No_Conflict", relations)
    res.to_csv(f"significance_{ct}_vs_NoConflict.csv", index=False)
    
    #print(res.round({"p": 5, "p_corrected": 5}).to_string(index=False))
    res_round = res.round({"p": 5, "p_corrected": 5})
    display(res.round({"p": 5, "p_corrected": 5}))


===== Challenge vs. No_Conflict =====


,relation,n_Challenge,%_Challenge,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,15,18.29,195,4.91,13.38,4.33,0.00001,0.00037,***
1,background,1,1.22,54,1.36,-0.14,0.90,1.00000,1.00000,n.s.
2,enablement,0,0.00,5,0.13,-0.13,0.00,1.00000,1.00000,n.s.
3,evaluation,0,0.00,61,1.54,-1.54,0.00,0.63589,1.00000,n.s.
4,evidence,1,1.22,39,0.98,0.24,1.24,0.56044,1.00000,n.s.
5,motivation,0,0.00,11,0.28,-0.28,0.00,1.00000,1.00000,n.s.
6,reason,0,0.00,53,1.34,-1.34,0.00,0.62730,1.00000,n.s.
7,antithesis,4,4.88,33,0.83,4.05,6.12,0.00620,0.14876,n.s.
8,concession,1,1.22,59,1.49,-0.27,0.82,1.00000,1.00000,n.s.
9,contrast,2,2.44,8,0.20,2.24,12.38,0.01639,0.37705,n.s.



===== Correction vs. No_Conflict =====


,relation,n_Correction,%_Correction,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,9,6.29,195,4.91,1.38,1.30,0.43089,1.00000,n.s.
1,background,0,0.00,54,1.36,-1.36,0.00,0.26288,1.00000,n.s.
2,enablement,0,0.00,5,0.13,-0.13,0.00,1.00000,1.00000,n.s.
3,evaluation,1,0.70,61,1.54,-0.84,0.45,0.72460,1.00000,n.s.
4,evidence,1,0.70,39,0.98,-0.28,0.71,1.00000,1.00000,n.s.
5,motivation,0,0.00,11,0.28,-0.28,0.00,1.00000,1.00000,n.s.
6,reason,0,0.00,53,1.34,-1.34,0.00,0.26239,1.00000,n.s.
7,antithesis,5,3.50,33,0.83,2.67,4.32,0.00944,0.24538,n.s.
8,concession,1,0.70,59,1.49,-0.79,0.47,0.72330,1.00000,n.s.
9,contrast,2,1.40,8,0.20,1.20,7.02,0.04500,1.00000,n.s.



===== Direct_NegEval vs. No_Conflict =====


,relation,n_Direct_NegEval,%_Direct_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,22,2.82,195,4.91,-2.10,0.56,0.00856,0.20541,n.s.
1,background,3,0.38,54,1.36,-0.98,0.28,0.01842,0.42370,n.s.
2,enablement,2,0.26,5,0.13,0.13,2.04,0.32388,1.00000,n.s.
3,evaluation,16,2.05,61,1.54,0.51,1.34,0.28081,1.00000,n.s.
4,evidence,4,0.51,39,0.98,-0.47,0.52,0.29880,1.00000,n.s.
5,motivation,0,0.00,11,0.28,-0.28,0.00,0.22986,1.00000,n.s.
6,reason,11,1.41,53,1.34,0.07,1.06,0.86511,1.00000,n.s.
7,antithesis,6,0.77,33,0.83,-0.06,0.92,1.00000,1.00000,n.s.
8,concession,17,2.18,59,1.49,0.69,1.47,0.16096,1.00000,n.s.
9,contrast,8,1.02,8,0.20,0.82,5.12,0.00193,0.04814,*



===== Indirect_NegEval vs. No_Conflict =====


,relation,n_Indirect_NegEval,%_Indirect_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,17,3.81,195,4.91,-1.10,0.77,0.35040,1.00000,n.s.
1,background,0,0.00,54,1.36,-1.36,0.00,0.00526,0.13157,n.s.
2,enablement,0,0.00,5,0.13,-0.13,0.00,1.00000,1.00000,n.s.
3,evaluation,3,0.67,61,1.54,-0.86,0.43,0.20654,1.00000,n.s.
4,evidence,5,1.12,39,0.98,0.14,1.14,0.79944,1.00000,n.s.
5,motivation,0,0.00,11,0.28,-0.28,0.00,0.61648,1.00000,n.s.
6,reason,4,0.90,53,1.34,-0.44,0.67,0.65580,1.00000,n.s.
7,antithesis,4,0.90,33,0.83,0.07,1.08,0.78504,1.00000,n.s.
8,concession,6,1.35,59,1.49,-0.14,0.90,1.00000,1.00000,n.s.
9,contrast,2,0.45,8,0.20,0.25,2.23,0.26781,1.00000,n.s.


## Part 3: paragraph-level analysis (Conflict types only)

   1. Every paragraph is assigned ONE type, depending on which Conflict type it contains.  
    If a paragraph contains several types, the first in this priority order wins:  
    Direct_NegEval > Indirect_NegEval > Challenge > Correction > No_Conflict
    
   2. For every EDU in the paragraph, all relations in its column  
    'rstree_relation_chain_subtree' are counted (one count per relation occurrence).  
     The unit counted here is therefore a RELATION OCCURRENCE inside a paragraph, not an EDU.  


In [27]:
# %% Step 10: build one long table with one row per relation occurrence
import ast

priority = ["Direct_NegEval", "Indirect_NegEval", "Challenge", "Correction", "No_Conflict"]

records = []        # (paragraph_id, paragraph_type, relation)
para_type_of = {}   # paragraph_id -> paragraph_type

for para_id, df_para in df.groupby("paragraph_id_consecutive"):
    present = set(df_para["Conflict_Type"])
    para_type = next((ct for ct in priority if ct in present), None)
    if para_type is None:
        continue
    para_type_of[para_id] = para_type

    for chain_str in df_para["rstree_relation_chain_subtree"]:
        for rel in ast.literal_eval(chain_str):   # string "['a', 'b', None]" -> Python list
            if rel is not None:                    # the root of the tree has no relation
                records.append((para_id, para_type, rel))

df_para_rel = pd.DataFrame(records, columns=["paragraph_id", "paragraph_type", "relation"])

# merge -N / -S variants, exactly as for the leaf analysis
df_para_rel["relation"] = df_para_rel["relation"].replace({
    "reason-N": "reason", "reason-S": "reason",
    "evaluation-N": "evaluation", "evaluation-S": "evaluation",
})

In [29]:
# %% Step 11: how many paragraphs and relation occurrences per type?
print("Paragraphs per type:\n")
paragraphs_per_types = pd.Series(para_type_of).value_counts()
paragraphs_per_types




Paragraphs per type:



No_Conflict         332
Direct_NegEval      200
Indirect_NegEval     71
Challenge            12
Correction            2
dtype: int64

In [30]:
print("Relation occurrences per type:\n" )
relation_occurences_per_type = df_para_rel["paragraph_type"].value_counts()
relation_occurences_per_type

Relation occurrences per type:



No_Conflict         14576
Direct_NegEval      13061
Indirect_NegEval     3369
Challenge            1054
Correction             49
Name: paragraph_type, dtype: int64

In [31]:
# %% Step 12: absolute numbers, proportions, percentages (same function as in Part 1)
# The denominator is ALL relation occurrences of that paragraph type (including span,
# sameunit, ...), so the percentages match your paragraph-level barplot.
table_para, totals_para = label_table(df_para_rel, "paragraph_type", relations)
#print(table_para)
table_para.to_csv("label_counts_paragraph_conflict_types.csv")
table_para

n                                                        proportion                                                          percent                                                       
paragraph_type Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict  Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict
relation                                                                                                                                                                                                          
attribution           21          1            121               38         138     0.0199     0.0204         0.0093           0.0113      0.0095      1.99       2.04           0.93             1.13        0.95
background            40          2            518               90         496     0.0380     0.0408         0.0397           0.0267      0.0340      3.80       4.08           3.97             2.67        3.40
enablement             4          0              9                5          10     0.0038     0.0000         0.0007           0.0015      0.0007      0.38       0.00           0.07             0.15        0.07
evaluation            14          0            164               48         157     0.0133     0.0000         0.0126           0.0142      0.0108      1.33       0.00           1.26             1.42        1.08
evidence              17          0            297               76         255     0.0161     0.0000         0.0227           0.0226      0.0175      1.61       0.00           2.27             2.26        1.75
motivation             0          0             86               20         158     0.0000     0.0000         0.0066           0.0059      0.0108      0.00       0.00           0.66             0.59        1.08
reason                26          0            375               78         443     0.0247     0.0000         0.0287           0.0232      0.0304      2.47       0.00           2.87             2.32        3.04
antithesis            18          6             81                7          59     0.0171     0.1224         0.0062           0.0021      0.0040      1.71      12.24           0.62             0.21        0.40
concession             3          0            154               33         192     0.0028     0.0000         0.0118           0.0098      0.0132      0.28       0.00           1.18             0.98        1.32
contrast              30          0            230               21          84     0.0285     0.0000         0.0176           0.0062      0.0058      2.85       0.00           1.76             0.62        0.58
cause                  0          1             87               17          50     0.0000     0.0204         0.0067           0.0050      0.0034      0.00       2.04           0.67             0.50        0.34
circumstance           3          0             95               44         141     0.0028     0.0000         0.0073           0.0131      0.0097      0.28       0.00           0.73             1.31        0.97
condition              0          0             44                8          47     0.0000     0.0000         0.0034           0.0024      0.0032      0.00       0.00           0.34             0.24        0.32
e-elaboration         38          3            463              136         486     0.0361     0.0612         0.0354           0.0404      0.0333      3.61       6.12           3.54             4.04        3.33
elaboration           44          0            522              155         712     0.0417     0.0000         0.0400           0.0460      0.0488      4.17       0.00           4.00             4.60        4.88
interpretation         3          0             84               15          65     0.0028     0.0000         0.0064           0.0045      0.0045      0.28       0.00           0.64             0.45        0.45
means         

In [32]:
# %% Step 13: each Conflict type vs. No_Conflict paragraphs (Fisher's exact + Holm)
for ct in conflict_types:
    print(f"\n===== Paragraphs with {ct} vs. No_Conflict paragraphs =====")
    res = compare_two_groups(df_para_rel, "paragraph_type", ct, "No_Conflict", relations)
    #print(res.round({"p": 5, "p_corrected": 5}).to_string(index=False))
    res.to_csv(f"significance_paragraph_{ct}_vs_NoConflict.csv", index=False)
    
    display(res.round({"p": 5, "p_corrected": 5}))


===== Paragraphs with Challenge vs. No_Conflict paragraphs =====


,relation,n_Challenge,%_Challenge,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,21,1.99,138,0.95,1.05,2.13,0.00348,0.07309,n.s.
1,background,40,3.80,496,3.40,0.39,1.12,0.48314,1.00000,n.s.
2,enablement,4,0.38,10,0.07,0.31,5.55,0.01192,0.23843,n.s.
3,evaluation,14,1.33,157,1.08,0.25,1.24,0.44136,1.00000,n.s.
4,evidence,17,1.61,255,1.75,-0.14,0.92,0.90265,1.00000,n.s.
5,motivation,0,0.00,158,1.08,-1.08,0.00,0.00004,0.00084,***
6,reason,26,2.47,443,3.04,-0.57,0.81,0.34906,1.00000,n.s.
7,antithesis,18,1.71,59,0.40,1.30,4.28,0.00000,0.00007,***
8,concession,3,0.28,192,1.32,-1.03,0.21,0.00133,0.02915,*
9,contrast,30,2.85,84,0.58,2.27,5.05,0.00000,0.00000,***



===== Paragraphs with Correction vs. No_Conflict paragraphs =====


,relation,n_Correction,%_Correction,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,1,2.04,138,0.95,1.09,2.18,0.37419,1.0,n.s.
1,background,2,4.08,496,3.40,0.68,1.21,0.68350,1.0,n.s.
2,enablement,0,0.00,10,0.07,-0.07,0.00,1.00000,1.0,n.s.
3,evaluation,0,0.00,157,1.08,-1.08,0.00,1.00000,1.0,n.s.
4,evidence,0,0.00,255,1.75,-1.75,0.00,1.00000,1.0,n.s.
5,motivation,0,0.00,158,1.08,-1.08,0.00,1.00000,1.0,n.s.
6,reason,0,0.00,443,3.04,-3.04,0.00,0.40593,1.0,n.s.
7,antithesis,6,12.24,59,0.40,11.84,34.33,0.00000,0.0,***
8,concession,0,0.00,192,1.32,-1.32,0.00,1.00000,1.0,n.s.
9,contrast,0,0.00,84,0.58,-0.58,0.00,1.00000,1.0,n.s.



===== Paragraphs with Direct_NegEval vs. No_Conflict paragraphs =====


,relation,n_Direct_NegEval,%_Direct_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,121,0.93,138,0.95,-0.02,0.98,0.90053,1.00000,n.s.
1,background,518,3.97,496,3.40,0.56,1.17,0.01358,0.23091,n.s.
2,enablement,9,0.07,10,0.07,0.00,1.00,1.00000,1.00000,n.s.
3,evaluation,164,1.26,157,1.08,0.18,1.17,0.17721,1.00000,n.s.
4,evidence,297,2.27,255,1.75,0.52,1.31,0.00192,0.03459,*
5,motivation,86,0.66,158,1.08,-0.43,0.60,0.00018,0.00369,**
6,reason,375,2.87,443,3.04,-0.17,0.94,0.41384,1.00000,n.s.
7,antithesis,81,0.62,59,0.40,0.22,1.54,0.01362,0.23091,n.s.
8,concession,154,1.18,192,1.32,-0.14,0.89,0.32943,1.00000,n.s.
9,contrast,230,1.76,84,0.58,1.18,3.09,0.00000,0.00000,***



===== Paragraphs with Indirect_NegEval vs. No_Conflict paragraphs =====


,relation,n_Indirect_NegEval,%_Indirect_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,attribution,38,1.13,138,0.95,0.18,1.19,0.33245,1.00000,n.s.
1,background,90,2.67,496,3.40,-0.73,0.78,0.03138,0.59626,n.s.
2,enablement,5,0.15,10,0.07,0.08,2.16,0.17816,1.00000,n.s.
3,evaluation,48,1.42,157,1.08,0.35,1.33,0.08792,1.00000,n.s.
4,evidence,76,2.26,255,1.75,0.51,1.30,0.05482,0.93194,n.s.
5,motivation,20,0.59,158,1.08,-0.49,0.54,0.00889,0.18665,n.s.
6,reason,78,2.32,443,3.04,-0.72,0.76,0.02279,0.45578,n.s.
7,antithesis,7,0.21,59,0.40,-0.20,0.51,0.11245,1.00000,n.s.
8,concession,33,0.98,192,1.32,-0.34,0.74,0.12195,1.00000,n.s.
9,contrast,21,0.62,84,0.58,0.05,1.08,0.70809,1.00000,n.s.


## Part 4: analysis per function group instead of per single relation
 Every relation is replaced by its function group (pragmatic, semantic, ...).
 Then the same counts and tests as before are run, now with only 6 groups instead of ~26 relations. Done for the leaf level and the paragraph level.

In [33]:
# %% Step 15: mapping relation -> function group
# Names are lowercase, to match the corpus after .str.lower()
# (the corpus has e.g. 'reason-N', 'evaluation-S', 'sameunit').
function_groups = {
    "pragmatic": ["background", "evidence", "reason", "reason-n", "reason-s",
                  "evaluation", "evaluation-n", "evaluation-s",
                  "motivation", "enablement"],
    "semantic": ["circumstance", "condition", "elaboration", "e-elaboration",
                 "interpretation", "cause", "result", "purpose",
                 "solutionhood", "unless"],
    "contrastive": ["antithesis", "concession", "contrast"],
    "textual": ["preparation", "restatement", "summary",
                "textual-organization", "topic-comment"],
    "multinuclear": ["same-unit", "sameunit", "sequence", "list",
                     "conjunction", "joint"],
    "other": ["attribution"],
}

# invert it: relation name -> group name
relation_to_function = {rel: group
                        for group, rels in function_groups.items()
                        for rel in rels}

function_names = list(function_groups.keys())   # order of the tables

def map_to_function(relation_series):
    """lowercase the relation names, map them to their group.
    Relations without a group (e.g. 'span', 'otherwise') become 'unmapped'.
    They are NOT shown in the tables, but stay in the denominators,
    exactly like span/sameunit etc. in the single-relation tables."""
    return relation_series.str.lower().map(relation_to_function).fillna("unmapped")

In [36]:
# %% Step 16: build the leaf-level and paragraph-level tables with function groups
# leaf level: one row per EDU (uses the raw leaf relation of the corpus)
df_func = df[["Conflict_Type", "Conflict_Binary"]].copy()
df_func["relation"] = map_to_function(df["rstree_relation_leaf"])

# paragraph level: one row per relation occurrence (from Step 10)
df_para_func = df_para_rel[["paragraph_type"]].copy()
df_para_func["relation"] = map_to_function(df_para_rel["relation"])

# check: which relation names were left out? (expect span, otherwise, ...)
unmapped_leaf = df.loc[df_func["relation"] == "unmapped", "rstree_relation_leaf"]
unmapped_para = df_para_rel.loc[df_para_func["relation"] == "unmapped", "relation"]
print("Unmapped at leaf level:\n")
unmapped_leaf.value_counts()


Unmapped at leaf level:

Unmapped at paragraph level:



span         18168
means           35
otherwise        7
Name: relation, dtype: int64

In [35]:
print("Unmapped at paragraph level:\n")
unmapped_para.value_counts()

Unmapped at paragraph level:



span         18168
means           35
otherwise        7
Name: relation, dtype: int64

### 4a) Leaf level (unit: EDU)

In [38]:
# %% Step 17: counts, proportions, percentages per function group
# five labels
table_func_types, totals = label_table(df_func, "Conflict_Type", function_names)

table_func_types.to_csv("function_counts_conflict_types.csv")
table_func_types

n                                                        proportion                                                          percent                                                       
Conflict_Type Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict  Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict
relation                                                                                                                                                                                                         
pragmatic             2          2             36               12         223     0.0244     0.0140         0.0461           0.0269      0.0562      2.44       1.40           4.61             2.69        5.62
semantic             16         32            193              114         913     0.1951     0.2238         0.2471           0.2556      0.2300     19.51      22.38          24.71            25.56       23.00
contrastive           7          8             31               12         100     0.0854     0.0559         0.0397           0.0269      0.0252      8.54       5.59           3.97             2.69        2.52
textual               2          2              4                1         114     0.0244     0.0140         0.0051           0.0022      0.0287      2.44       1.40           0.51             0.22        2.87
multinuclear          8         32            189              119         966     0.0976     0.2238         0.2420           0.2668      0.2434      9.76      22.38          24.20            26.68       24.34
other                15          9             22               17         195     0.1829     0.0629         0.0282           0.0381      0.0491     18.29       6.29           2.82             3.81        4.91

In [40]:
# Conflict vs. No Conflict
table_func_binary, totals = label_table(df_func, "Conflict_Binary", function_names)
table_func_binary.to_csv("function_counts_binary.csv")
table_func_binary

n             proportion              percent            
Conflict_Binary Conflict No_Conflict   Conflict No_Conflict Conflict No_Conflict
relation                                                                        
pragmatic             52         223     0.0358      0.0562     3.58        5.62
semantic             355         913     0.2445      0.2300    24.45       23.00
contrastive           58         100     0.0399      0.0252     3.99        2.52
textual                9         114     0.0062      0.0287     0.62        2.87
multinuclear         348         966     0.2397      0.2434    23.97       24.34
other                 63         195     0.0434      0.0491     4.34        4.91

In [42]:
# %% Step 18: significance tests per function group (leaf level)
# Conflict vs. No Conflict
res_func_binary = compare_two_groups(df_func, "Conflict_Binary", "Conflict",
                                     "No_Conflict", function_names)
#print(res_func_binary.round({"p": 5, "p_corrected": 5}).to_string(index=False))

res_func_binary.to_csv("function_significance_binary.csv", index=False)
display(res_func_binary.round({"p": 5, "p_corrected": 5}))

,relation,n_Conflict,%_Conflict,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,52,3.58,223,5.62,-2.04,0.62,0.00207,0.01034,*
1,semantic,355,24.45,913,23.00,1.45,1.08,0.27711,0.83134,n.s.
2,contrastive,58,3.99,100,2.52,1.47,1.61,0.00602,0.02409,*
3,textual,9,0.62,114,2.87,-2.25,0.21,0.00000,0.00000,***
4,multinuclear,348,23.97,966,24.34,-0.37,0.98,0.80222,0.85619,n.s.
5,other,63,4.34,195,4.91,-0.57,0.88,0.42809,0.85619,n.s.


In [49]:
print(res_func_binary.round({"p": 5, "p_corrected": 5}).to_string(index=False))

    relation  n_Conflict  %_Conflict  n_No_Conflict  %_No_Conflict  diff_pct_points  odds_ratio       p  p_corrected  sig
   pragmatic          52        3.58            223           5.62            -2.04        0.62 0.00207      0.01034    *
    semantic         355       24.45            913          23.00             1.45        1.08 0.27711      0.83134 n.s.
 contrastive          58        3.99            100           2.52             1.47        1.61 0.00602      0.02409    *
     textual           9        0.62            114           2.87            -2.25        0.21 0.00000      0.00000  ***
multinuclear         348       23.97            966          24.34            -0.37        0.98 0.80222      0.85619 n.s.
       other          63        4.34            195           4.91            -0.57        0.88 0.42809      0.85619 n.s.


In [43]:
# each Conflict type vs. No Conflict
for ct in conflict_types:
    print(f"\n===== Function groups: {ct} vs. No_Conflict (leaf level) =====")
    res = compare_two_groups(df_func, "Conflict_Type", ct, "No_Conflict", function_names)
    #print(res.round({"p": 5, "p_corrected": 5}).to_string(index=False))
    
    res.to_csv(f"function_significance_{ct}_vs_NoConflict.csv", index=False)
    display(res.round({"p": 5, "p_corrected": 5}))


===== Function groups: Challenge vs. No_Conflict (leaf level) =====


,relation,n_Challenge,%_Challenge,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,2,2.44,223,5.62,-3.18,0.42,0.32536,0.97609,n.s.
1,semantic,16,19.51,913,23.00,-3.49,0.81,0.50918,1.00000,n.s.
2,contrastive,7,8.54,100,2.52,6.02,3.61,0.00550,0.02199,*
3,textual,2,2.44,114,2.87,-0.43,0.85,1.00000,1.00000,n.s.
4,multinuclear,8,9.76,966,24.34,-14.58,0.34,0.00152,0.00759,**
5,other,15,18.29,195,4.91,13.38,4.33,0.00001,0.00009,***



===== Function groups: Correction vs. No_Conflict (leaf level) =====


,relation,n_Correction,%_Correction,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,2,1.40,223,5.62,-4.22,0.24,0.02340,0.14040,n.s.
1,semantic,32,22.38,913,23.00,-0.63,0.96,0.91965,1.00000,n.s.
2,contrastive,8,5.59,100,2.52,3.07,2.29,0.05388,0.26942,n.s.
3,textual,2,1.40,114,2.87,-1.47,0.48,0.43902,1.00000,n.s.
4,multinuclear,32,22.38,966,24.34,-1.96,0.90,0.69123,1.00000,n.s.
5,other,9,6.29,195,4.91,1.38,1.30,0.43089,1.00000,n.s.



===== Function groups: Direct_NegEval vs. No_Conflict (leaf level) =====


,relation,n_Direct_NegEval,%_Direct_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,36,4.61,223,5.62,-1.01,0.81,0.30050,0.90150,n.s.
1,semantic,193,24.71,913,23.00,1.71,1.10,0.30834,0.90150,n.s.
2,contrastive,31,3.97,100,2.52,1.45,1.60,0.03068,0.12270,n.s.
3,textual,4,0.51,114,2.87,-2.36,0.17,0.00001,0.00008,***
4,multinuclear,189,24.20,966,24.34,-0.14,0.99,0.96362,0.96362,n.s.
5,other,22,2.82,195,4.91,-2.10,0.56,0.00856,0.04279,*



===== Function groups: Indirect_NegEval vs. No_Conflict (leaf level) =====


,relation,n_Indirect_NegEval,%_Indirect_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,12,2.69,223,5.62,-2.93,0.46,0.00724,0.03619,*
1,semantic,114,25.56,913,23.00,2.56,1.15,0.23709,0.94837,n.s.
2,contrastive,12,2.69,100,2.52,0.17,1.07,0.75218,0.94837,n.s.
3,textual,1,0.22,114,2.87,-2.65,0.08,0.00011,0.00066,***
4,multinuclear,119,26.68,966,24.34,2.34,1.13,0.27110,0.94837,n.s.
5,other,17,3.81,195,4.91,-1.10,0.77,0.35040,0.94837,n.s.


### 4b) Paragraph level (unit: relation occurrence in a paragraph)
Conflict types only, as before.

In [45]:
# %% Step 20: counts, proportions, percentages per function group
table_func_para, totals = label_table(df_para_func, "paragraph_type", function_names)

table_func_para.to_csv("function_counts_paragraph_conflict_types.csv")
table_func_para

n                                                        proportion                                                          percent                                                       
paragraph_type Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict  Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict Challenge Correction Direct_NegEval Indirect_NegEval No_Conflict
relation                                                                                                                                                                                                          
pragmatic            101          2           1449              317        1519     0.0958     0.0408         0.1109           0.0941      0.1042      9.58       4.08          11.09             9.41       10.42
semantic              97          4           1497              443        1773     0.0920     0.0816         0.1146           0.1315      0.1216      9.20       8.16          11.46            13.15       12.16
contrastive           51          6            465               61         335     0.0484     0.1224         0.0356           0.0181      0.0230      4.84      12.24           3.56             1.81        2.30
textual               12          1            117               24         213     0.0114     0.0204         0.0090           0.0071      0.0146      1.14       2.04           0.90             0.71        1.46
multinuclear         160          4           2044              591        2294     0.1518     0.0816         0.1565           0.1754      0.1574     15.18       8.16          15.65            17.54       15.74
other                 21          1            121               38         138     0.0199     0.0204         0.0093           0.0113      0.0095      1.99       2.04           0.93             1.13        0.95

In [46]:
# %% Step 21: significance tests per function group (paragraph level)
for ct in conflict_types:
    print(f"\n===== Function groups: paragraphs with {ct} vs. No_Conflict =====")
    res = compare_two_groups(df_para_func, "paragraph_type", ct, "No_Conflict", function_names)
    #print(res.round({"p": 5, "p_corrected": 5}).to_string(index=False))
    
    res.to_csv(f"function_significance_paragraph_{ct}_vs_NoConflict.csv", index=False)
    display(res.round({"p": 5, "p_corrected": 5}))


===== Function groups: paragraphs with Challenge vs. No_Conflict =====


,relation,n_Challenge,%_Challenge,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,101,9.58,1519,10.42,-0.84,0.91,0.43235,1.00000,n.s.
1,semantic,97,9.20,1773,12.16,-2.96,0.73,0.00370,0.01740,*
2,contrastive,51,4.84,335,2.30,2.54,2.16,0.00000,0.00003,***
3,textual,12,1.14,213,1.46,-0.32,0.78,0.50192,1.00000,n.s.
4,multinuclear,160,15.18,2294,15.74,-0.56,0.96,0.66117,1.00000,n.s.
5,other,21,1.99,138,0.95,1.05,2.13,0.00348,0.01740,*



===== Function groups: paragraphs with Correction vs. No_Conflict =====


,relation,n_Correction,%_Correction,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,2,4.08,1519,10.42,-6.34,0.37,0.23496,0.93983,n.s.
1,semantic,4,8.16,1773,12.16,-4.00,0.64,0.51331,1.00000,n.s.
2,contrastive,6,12.24,335,2.30,9.95,5.93,0.00092,0.00554,**
3,textual,1,2.04,213,1.46,0.58,1.40,0.51494,1.00000,n.s.
4,multinuclear,4,8.16,2294,15.74,-7.57,0.48,0.17092,0.85461,n.s.
5,other,1,2.04,138,0.95,1.09,2.18,0.37419,1.00000,n.s.



===== Function groups: paragraphs with Direct_NegEval vs. No_Conflict =====


,relation,n_Direct_NegEval,%_Direct_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,1449,11.09,1519,10.42,0.67,1.07,0.07343,0.29340,n.s.
1,semantic,1497,11.46,1773,12.16,-0.70,0.93,0.07335,0.29340,n.s.
2,contrastive,465,3.56,335,2.30,1.26,1.57,0.00000,0.00000,***
3,textual,117,0.90,213,1.46,-0.57,0.61,0.00001,0.00007,***
4,multinuclear,2044,15.65,2294,15.74,-0.09,0.99,0.84249,1.00000,n.s.
5,other,121,0.93,138,0.95,-0.02,0.98,0.90053,1.00000,n.s.



===== Function groups: paragraphs with Indirect_NegEval vs. No_Conflict =====


,relation,n_Indirect_NegEval,%_Indirect_NegEval,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,317,9.41,1519,10.42,-1.01,0.89,0.08280,0.33118,n.s.
1,semantic,443,13.15,1773,12.16,0.99,1.09,0.11678,0.33118,n.s.
2,contrastive,61,1.81,335,2.30,-0.49,0.78,0.09042,0.33118,n.s.
3,textual,24,0.71,213,1.46,-0.75,0.48,0.00030,0.00180,**
4,multinuclear,591,17.54,2294,15.74,1.80,1.14,0.01076,0.05382,n.s.
5,other,38,1.13,138,0.95,0.18,1.19,0.33245,0.33245,n.s.


In [48]:
# %% Step 22: significance tests per function group (paragraph level)
# Conflict vs. No Conflict

# 1) create the binary paragraph label: a paragraph is "Conflict" if it was assigned
#    one of the four Conflict types in Step 10, otherwise "No_Conflict"
df_para_func["Conflict_Binary"] = df_para_func["paragraph_type"].apply(
    lambda t: "No_Conflict" if t == "No_Conflict" else "Conflict")

# check: how many relation occurrences per group?
print(df_para_func["Conflict_Binary"].value_counts(), "\n")

# 2) counts, proportions, percentages (same function as before)
table_func_para_binary, totals = label_table(df_para_func, "Conflict_Binary", function_names)
display(table_func_para_binary)
table_func_para_binary.to_csv("function_counts_paragraph_binary.csv")

# 3) significance tests: Fisher's exact test per function group, Holm-corrected
res_func_para_binary = compare_two_groups(df_para_func, "Conflict_Binary", "Conflict",
                                          "No_Conflict", function_names)
res_func_para_binary.to_csv("function_significance_paragraph_binary.csv", index=False)
display(res_func_para_binary.round({"p": 5, "p_corrected": 5}))

Conflict       17533
No_Conflict    14576
Name: Conflict_Binary, dtype: int64 



n             proportion              percent            
Conflict_Binary Conflict No_Conflict   Conflict No_Conflict Conflict No_Conflict
relation                                                                        
pragmatic           1869        1519     0.1066      0.1042    10.66       10.42
semantic            2041        1773     0.1164      0.1216    11.64       12.16
contrastive          583         335     0.0333      0.0230     3.33        2.30
textual              154         213     0.0088      0.0146     0.88        1.46
multinuclear        2799        2294     0.1596      0.1574    15.96       15.74
other                181         138     0.0103      0.0095     1.03        0.95

,relation,n_Conflict,%_Conflict,n_No_Conflict,%_No_Conflict,diff_pct_points,odds_ratio,p,p_corrected,sig
0,pragmatic,1869,10.66,1519,10.42,0.24,1.03,0.49969,1.00000,n.s.
1,semantic,2041,11.64,1773,12.16,-0.52,0.95,0.15052,0.60208,n.s.
2,contrastive,583,3.33,335,2.30,1.03,1.46,0.00000,0.00000,***
3,textual,154,0.88,213,1.46,-0.58,0.60,0.00000,0.00001,***
4,multinuclear,2799,15.96,2294,15.74,0.23,1.02,0.59131,1.00000,n.s.
5,other,181,1.03,138,0.95,0.09,1.09,0.46276,1.00000,n.s.


In [51]:
display(res_func_para_binary.round({"p": 5, "p_corrected": 5}).to_string(index=False))

'    relation  n_Conflict  %_Conflict  n_No_Conflict  %_No_Conflict  diff_pct_points  odds_ratio       p  p_corrected  sig\n   pragmatic        1869       10.66           1519          10.42             0.24        1.03 0.49969      1.00000 n.s.\n    semantic        2041       11.64           1773          12.16            -0.52        0.95 0.15052      0.60208 n.s.\n contrastive         583        3.33            335           2.30             1.03        1.46 0.00000      0.00000  ***\n     textual         154        0.88            213           1.46            -0.58        0.60 0.00000      0.00001  ***\nmultinuclear        2799       15.96           2294          15.74             0.23        1.02 0.59131      1.00000 n.s.\n       other         181        1.03            138           0.95             0.09        1.09 0.46276      1.00000 n.s.'